[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 中等: 线性回归

使用三种不同方法实现**线性回归**——全部使用纯 PyTorch。

给定形状为 `(N, D)` 的数据 `X` 和形状为 `(N,)` 的目标 `y`，找到形状为 `(D,)` 的权重 `w` 和偏置 `b`（标量），使得：

$$\hat{y} = Xw + b$$

### 函数签名
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

所有方法返回 `(w, b)`，其中 `w` 的形状为 `(D,)`，`b` 的形状为 `()`。

### 方法 1 — 闭式解法（正规方程）
用全 1 列扩展 X，然后求解：

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

或者使用 `torch.linalg.lstsq` / `torch.linalg.solve`。

### 方法 2 — 从头实现梯度下降
初始化 `w` 和 `b` 为零。重复 `steps` 次迭代：
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### 方法 3 — PyTorch nn.Linear
创建 `nn.Linear(D, 1)`，使用 `nn.MSELoss` 和优化器（如 `torch.optim.SGD`）。训练后，从层中提取 `w` 和 `b`。

### 规则
- 所有输入和输出必须是 **PyTorch 张量**
- 不要使用 numpy 或 sklearn
- `closed_form` 不得使用迭代优化
- `gradient_descent` 必须手动计算梯度（不使用 `autograd`）
- `nn_linear` 应该使用 `torch.nn.Linear` 和 `损失.backward()`

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ 在此实现你的代码

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """正规方程: w = (X^T X)^{-1} X^T y"""
        pass  # 返回 (w, b)

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):
        """手动梯度下降循环"""
        pass  # 返回 (w, b)

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """使用 autograd 的 nn.Linear 训练"""
        pass  # 返回 (w, b)

In [ ]:
import torch.optim as optim

class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """
        使用正规方程求解线性回归的闭式解
        """
        # 添加全1列到X，形成增广矩阵 X_aug
        N = X.shape[0]
        ones = torch.ones(N, 1)
        X_aug = torch.cat([ones, X], dim=1)  # 形状 (N, D+1)
        
        # 使用torch.linalg.lstsq求解 X_aug @ theta = y
        theta, _ = torch.linalg.lstsq(X_aug, y.unsqueeze(1), rcond=None)
        
        # theta[0] 是偏置 b，theta[1:] 是权重 w
        b = theta[0].squeeze()
        w = theta[1:].squeeze()
        
        return w, b
    
    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor, lr=0.01, steps=1000) -> tuple[torch.Tensor, torch.Tensor]:
        """
        使用手动梯度下降求解线性回归
        """
        N, D = X.shape
        
        # 初始化权重和偏置为零
        w = torch.zeros(D)
        b = torch.tensor(0.0)
        
        for _ in range(steps):
            # 前向传播：计算预测值
            pred = X @ w + b  # 形状 (N,)
            
            # 计算误差
            error = pred - y  # 形状 (N,)
            
            # 手动计算梯度
            grad_w = (2.0 / N) * X.T @ error  # 形状 (D,)
            grad_b = (2.0 / N) * error.sum()   # 标量
            
            # 更新参数
            w -= lr * grad_w
            b -= lr * grad_b
        
        return w, b
    
    def nn_linear(self, X: torch.Tensor, y: torch.Tensor, lr=0.01, steps=1000) -> tuple[torch.Tensor, torch.Tensor]:
        """
        使用PyTorch的nn.Linear和优化器求解线性回归
        """
        N, D = X.shape
        
        # 创建线性层
        model = nn.Linear(D, 1, bias=True)
        
        # 定义损失函数和优化器
        criterion = nn.MSELoss()
        optimizer = optim.SGD(model.parameters(), lr=lr)
        
        # 确保y是列向量
        y = y.reshape(-1, 1)
        
        # 训练循环
        for _ in range(steps):
            # 前向传播
            pred = model(X)  # 形状 (N, 1)
            
            # 计算损失
            loss = criterion(pred, y)
            
            # 反向传播
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        # 提取权重和偏置
        w = model.weight.squeeze()  # 形状 (D,)
        b = model.bias.squeeze()    # 标量
        
        return w, b

In [ ]:
# 🧪 调试
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"闭式解法:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"梯度下降: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\n真实值:         w={true_w}, b=3.0")

In [ ]:
# ✅ 提交
from torch_judge import check
check("linear_regression")